# Find a real Attention U-Net advantage and export every stage

This searches your **existing saved models**, finds suitable cases if they exist, then downloads the comparison figures. **No models are retrained, no cutoffs are changed, and no heatmaps are edited.**

1. Open in Google Colab and select **Runtime → Change runtime type → T4 GPU**.
2. Run the single code cell and connect the same Drive account as your completed comparison.
3. Wait for the image ZIP to download. This scans multiple images, so it takes longer than the one-image notebook. If the original dataset is unavailable, the cell downloads the public test archive (about 245 MB) once. Completed search results are reused on reruns.

## Selection rule

The default search covers the **75 official-test melanoma cases**. A case qualifies only if Attention U-Net:

- **Correctly classifies the lesion** using its saved threshold;
- Achieves **Dice of at least 0.90**;
- Has **Dice at least 0.005 higher than every other pipeline**, with higher IoU as well.

Cases where another classifier is wrong are preferred, followed by the largest Dice lead. Up to **three examples** are exported and checked again using the original single-image inference. Choose **All test cases** to search all 379 official-test images, including benign lesions. If no case meets the rule, the notebook reports that and saves the search results.

## What you receive

For each selected image, every pipeline is shown in one row, with:

**Original → Ground truth → Segmentation probability → Predicted mask → Mask overlay → Isolated lesion → Actual classifier ROI → Grad-CAM heatmap → Grad-CAM overlay.**

The ZIP also includes two enlarged slide panels, one image per pipeline, separate stage PNGs, the actual predictions and all per-image search results.

## Why Attention and U-Net ROIs look different

- **Attention pipeline:** padded RGB lesion crop with some surrounding skin, square padding, and a dual-input classifier that also receives the full image.
- **U-Net pipeline:** predicted-mask background removal followed by a padded crop, giving a black background. Its classifier receives this single ROI.

Both are intentional saved preprocessing choices. Changing only the display would misrepresent the classifier input, so these differences are preserved. Grad-CAM explains the **melanoma class** in every row (Attention ROI branch). Each heatmap is independently normalised; stronger colours or a broader activation do not establish better performance. Grad-CAM appearance is **not** used to select cases.

**Presentation caption:** “Selected illustrative case where Attention U-Net correctly classified the lesion and achieved higher segmentation overlap than the other pipelines.”

This is a deliberately selected example, not evidence of overall superiority. The peer models retain their different training splits and ROI methods; some official-test images may have been used in peer development. This is not a common untouched test-set comparison. The supplied U-Net++ notebook implements standard U-Net and is labelled U-Net. Keep the overall evaluation results separate from these illustrations.


In [ ]:
#@title Search existing models for a genuine Attention U-Net advantage
SEARCH_SCOPE = "Melanoma only" #@param ["Melanoma only", "All test cases"]
MIN_ATTENTION_DICE = 0.90 #@param {type:"number"}
MIN_DICE_LEAD = 0.005 #@param {type:"number"}
NUMBER_OF_EXAMPLES = 3 #@param {type:"integer"}

from pathlib import Path
from datetime import datetime, timezone
import importlib
import sys
import pandas as pd
import tensorflow as tf
from IPython.display import display, Image as DisplayImage
from google.colab import drive, files

if not 0 <= MIN_ATTENTION_DICE <= 1 or not 0 < MIN_DICE_LEAD <= 1:
    raise ValueError('Dice must be between 0 and 1, and the required lead must be positive.')
if not 1 <= NUMBER_OF_EXAMPLES <= 5:
    raise ValueError('Choose 1 to 5 examples.')
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
COMPARISON_ROOT = DRIVE_ROOT / 'Melanoma_BestCase_Comparison_From_Notebooks'
PREFERRED_COMPARISON = COMPARISON_ROOT / 'comparisons/20261014T185422_968354Z'
SEARCH_ROOT = COMPARISON_ROOT / 'attention_advantage_search'
RUNTIME = Path('/content/melanoma_attention_advantage_search')
MODULE_DIR = RUNTIME / 'modules'
MODULE_DIR.mkdir(parents=True, exist_ok=True)
for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

LABELS = [{'image_id': 'ISIC_0000003', 'label': 0}, {'image_id': 'ISIC_0000012', 'label': 0}, {'image_id': 'ISIC_0000013', 'label': 1}, {'image_id': 'ISIC_0000014', 'label': 0}, {'image_id': 'ISIC_0000015', 'label': 0}, {'image_id': 'ISIC_0000020', 'label': 0}, {'image_id': 'ISIC_0000022', 'label': 1}, {'image_id': 'ISIC_0000023', 'label': 0}, {'image_id': 'ISIC_0000027', 'label': 0}, {'image_id': 'ISIC_0000036', 'label': 1}, {'image_id': 'ISIC_0000037', 'label': 0}, {'image_id': 'ISIC_0000040', 'label': 1}, {'image_id': 'ISIC_0000043', 'label': 1}, {'image_id': 'ISIC_0000052', 'label': 0}, {'image_id': 'ISIC_0000053', 'label': 0}, {'image_id': 'ISIC_0000056', 'label': 1}, {'image_id': 'ISIC_0000057', 'label': 0}, {'image_id': 'ISIC_0000064', 'label': 0}, {'image_id': 'ISIC_0000066', 'label': 0}, {'image_id': 'ISIC_0000069', 'label': 0}, {'image_id': 'ISIC_0000071', 'label': 0}, {'image_id': 'ISIC_0000072', 'label': 0}, {'image_id': 'ISIC_0000088', 'label': 0}, {'image_id': 'ISIC_0000092', 'label': 0}, {'image_id': 'ISIC_0000098', 'label': 0}, {'image_id': 'ISIC_0000099', 'label': 0}, {'image_id': 'ISIC_0000101', 'label': 0}, {'image_id': 'ISIC_0000107', 'label': 0}, {'image_id': 'ISIC_0000111', 'label': 0}, {'image_id': 'ISIC_0000113', 'label': 0}, {'image_id': 'ISIC_0000115', 'label': 0}, {'image_id': 'ISIC_0000117', 'label': 0}, {'image_id': 'ISIC_0000125', 'label': 0}, {'image_id': 'ISIC_0000126', 'label': 0}, {'image_id': 'ISIC_0000129', 'label': 0}, {'image_id': 'ISIC_0000132', 'label': 0}, {'image_id': 'ISIC_0000136', 'label': 0}, {'image_id': 'ISIC_0000138', 'label': 0}, {'image_id': 'ISIC_0000141', 'label': 1}, {'image_id': 'ISIC_0000149', 'label': 1}, {'image_id': 'ISIC_0000160', 'label': 1}, {'image_id': 'ISIC_0000161', 'label': 1}, {'image_id': 'ISIC_0000164', 'label': 1}, {'image_id': 'ISIC_0000165', 'label': 1}, {'image_id': 'ISIC_0000169', 'label': 1}, {'image_id': 'ISIC_0000172', 'label': 1}, {'image_id': 'ISIC_0000174', 'label': 1}, {'image_id': 'ISIC_0000177', 'label': 0}, {'image_id': 'ISIC_0000188', 'label': 0}, {'image_id': 'ISIC_0000195', 'label': 0}, {'image_id': 'ISIC_0000196', 'label': 0}, {'image_id': 'ISIC_0000197', 'label': 0}, {'image_id': 'ISIC_0000198', 'label': 0}, {'image_id': 'ISIC_0000200', 'label': 0}, {'image_id': 'ISIC_0000201', 'label': 0}, {'image_id': 'ISIC_0000202', 'label': 0}, {'image_id': 'ISIC_0000212', 'label': 0}, {'image_id': 'ISIC_0000213', 'label': 0}, {'image_id': 'ISIC_0000222', 'label': 0}, {'image_id': 'ISIC_0000226', 'label': 0}, {'image_id': 'ISIC_0000227', 'label': 0}, {'image_id': 'ISIC_0000228', 'label': 0}, {'image_id': 'ISIC_0000230', 'label': 0}, {'image_id': 'ISIC_0000231', 'label': 0}, {'image_id': 'ISIC_0000233', 'label': 0}, {'image_id': 'ISIC_0000234', 'label': 0}, {'image_id': 'ISIC_0000246', 'label': 0}, {'image_id': 'ISIC_0000253', 'label': 0}, {'image_id': 'ISIC_0000254', 'label': 0}, {'image_id': 'ISIC_0000257', 'label': 0}, {'image_id': 'ISIC_0000258', 'label': 0}, {'image_id': 'ISIC_0000271', 'label': 0}, {'image_id': 'ISIC_0000279', 'label': 1}, {'image_id': 'ISIC_0000287', 'label': 1}, {'image_id': 'ISIC_0000289', 'label': 1}, {'image_id': 'ISIC_0000298', 'label': 1}, {'image_id': 'ISIC_0000299', 'label': 1}, {'image_id': 'ISIC_0000302', 'label': 1}, {'image_id': 'ISIC_0000306', 'label': 1}, {'image_id': 'ISIC_0000310', 'label': 1}, {'image_id': 'ISIC_0000311', 'label': 1}, {'image_id': 'ISIC_0000319', 'label': 0}, {'image_id': 'ISIC_0000320', 'label': 0}, {'image_id': 'ISIC_0000325', 'label': 0}, {'image_id': 'ISIC_0000327', 'label': 0}, {'image_id': 'ISIC_0000328', 'label': 0}, {'image_id': 'ISIC_0000340', 'label': 0}, {'image_id': 'ISIC_0000343', 'label': 0}, {'image_id': 'ISIC_0000354', 'label': 0}, {'image_id': 'ISIC_0000357', 'label': 0}, {'image_id': 'ISIC_0000368', 'label': 0}, {'image_id': 'ISIC_0000371', 'label': 0}, {'image_id': 'ISIC_0000377', 'label': 0}, {'image_id': 'ISIC_0000378', 'label': 0}, {'image_id': 'ISIC_0000380', 'label': 0}, {'image_id': 'ISIC_0000388', 'label': 0}, {'image_id': 'ISIC_0000392', 'label': 0}, {'image_id': 'ISIC_0000393', 'label': 1}, {'image_id': 'ISIC_0000407', 'label': 0}, {'image_id': 'ISIC_0000418', 'label': 0}, {'image_id': 'ISIC_0000420', 'label': 1}, {'image_id': 'ISIC_0000433', 'label': 0}, {'image_id': 'ISIC_0000448', 'label': 0}, {'image_id': 'ISIC_0000449', 'label': 0}, {'image_id': 'ISIC_0000456', 'label': 0}, {'image_id': 'ISIC_0000466', 'label': 1}, {'image_id': 'ISIC_0000470', 'label': 0}, {'image_id': 'ISIC_0000476', 'label': 0}, {'image_id': 'ISIC_0000479', 'label': 0}, {'image_id': 'ISIC_0000482', 'label': 1}, {'image_id': 'ISIC_0000484', 'label': 1}, {'image_id': 'ISIC_0000487', 'label': 1}, {'image_id': 'ISIC_0000490', 'label': 0}, {'image_id': 'ISIC_0000494', 'label': 0}, {'image_id': 'ISIC_0000499', 'label': 0}, {'image_id': 'ISIC_0000501', 'label': 0}, {'image_id': 'ISIC_0000508', 'label': 0}, {'image_id': 'ISIC_0000509', 'label': 0}, {'image_id': 'ISIC_0000510', 'label': 0}, {'image_id': 'ISIC_0000515', 'label': 0}, {'image_id': 'ISIC_0000518', 'label': 1}, {'image_id': 'ISIC_0000525', 'label': 0}, {'image_id': 'ISIC_0000527', 'label': 0}, {'image_id': 'ISIC_0000533', 'label': 1}, {'image_id': 'ISIC_0000534', 'label': 0}, {'image_id': 'ISIC_0000537', 'label': 0}, {'image_id': 'ISIC_0000539', 'label': 0}, {'image_id': 'ISIC_0000540', 'label': 0}, {'image_id': 'ISIC_0000547', 'label': 1}, {'image_id': 'ISIC_0000549', 'label': 1}, {'image_id': 'ISIC_0000550', 'label': 1}, {'image_id': 'ISIC_0001100', 'label': 1}, {'image_id': 'ISIC_0001103', 'label': 1}, {'image_id': 'ISIC_0001128', 'label': 1}, {'image_id': 'ISIC_0001131', 'label': 1}, {'image_id': 'ISIC_0001142', 'label': 1}, {'image_id': 'ISIC_0001181', 'label': 0}, {'image_id': 'ISIC_0001185', 'label': 0}, {'image_id': 'ISIC_0001186', 'label': 0}, {'image_id': 'ISIC_0001190', 'label': 0}, {'image_id': 'ISIC_0001204', 'label': 0}, {'image_id': 'ISIC_0001242', 'label': 0}, {'image_id': 'ISIC_0001299', 'label': 0}, {'image_id': 'ISIC_0001427', 'label': 0}, {'image_id': 'ISIC_0001484', 'label': 0}, {'image_id': 'ISIC_0001685', 'label': 0}, {'image_id': 'ISIC_0001960', 'label': 0}, {'image_id': 'ISIC_0002107', 'label': 0}, {'image_id': 'ISIC_0002246', 'label': 0}, {'image_id': 'ISIC_0002673', 'label': 0}, {'image_id': 'ISIC_0002829', 'label': 0}, {'image_id': 'ISIC_0002871', 'label': 0}, {'image_id': 'ISIC_0003056', 'label': 0}, {'image_id': 'ISIC_0003559', 'label': 0}, {'image_id': 'ISIC_0003728', 'label': 0}, {'image_id': 'ISIC_0004115', 'label': 0}, {'image_id': 'ISIC_0004346', 'label': 0}, {'image_id': 'ISIC_0005000', 'label': 0}, {'image_id': 'ISIC_0006193', 'label': 0}, {'image_id': 'ISIC_0006711', 'label': 0}, {'image_id': 'ISIC_0007322', 'label': 0}, {'image_id': 'ISIC_0007693', 'label': 0}, {'image_id': 'ISIC_0008116', 'label': 0}, {'image_id': 'ISIC_0008207', 'label': 0}, {'image_id': 'ISIC_0008406', 'label': 0}, {'image_id': 'ISIC_0008507', 'label': 0}, {'image_id': 'ISIC_0008600', 'label': 0}, {'image_id': 'ISIC_0008626', 'label': 0}, {'image_id': 'ISIC_0008659', 'label': 0}, {'image_id': 'ISIC_0008992', 'label': 0}, {'image_id': 'ISIC_0008998', 'label': 0}, {'image_id': 'ISIC_0009035', 'label': 0}, {'image_id': 'ISIC_0009078', 'label': 0}, {'image_id': 'ISIC_0009083', 'label': 0}, {'image_id': 'ISIC_0009201', 'label': 0}, {'image_id': 'ISIC_0009298', 'label': 0}, {'image_id': 'ISIC_0009564', 'label': 0}, {'image_id': 'ISIC_0009869', 'label': 0}, {'image_id': 'ISIC_0009872', 'label': 0}, {'image_id': 'ISIC_0009874', 'label': 0}, {'image_id': 'ISIC_0009879', 'label': 0}, {'image_id': 'ISIC_0009880', 'label': 0}, {'image_id': 'ISIC_0009881', 'label': 0}, {'image_id': 'ISIC_0009882', 'label': 1}, {'image_id': 'ISIC_0009885', 'label': 0}, {'image_id': 'ISIC_0009889', 'label': 0}, {'image_id': 'ISIC_0009891', 'label': 0}, {'image_id': 'ISIC_0009898', 'label': 1}, {'image_id': 'ISIC_0009901', 'label': 0}, {'image_id': 'ISIC_0009902', 'label': 0}, {'image_id': 'ISIC_0009906', 'label': 0}, {'image_id': 'ISIC_0009918', 'label': 1}, {'image_id': 'ISIC_0009920', 'label': 0}, {'image_id': 'ISIC_0009923', 'label': 0}, {'image_id': 'ISIC_0009926', 'label': 0}, {'image_id': 'ISIC_0009927', 'label': 1}, {'image_id': 'ISIC_0009928', 'label': 0}, {'image_id': 'ISIC_0009930', 'label': 0}, {'image_id': 'ISIC_0009931', 'label': 0}, {'image_id': 'ISIC_0009943', 'label': 0}, {'image_id': 'ISIC_0009945', 'label': 0}, {'image_id': 'ISIC_0009946', 'label': 0}, {'image_id': 'ISIC_0009948', 'label': 0}, {'image_id': 'ISIC_0009954', 'label': 0}, {'image_id': 'ISIC_0009955', 'label': 1}, {'image_id': 'ISIC_0009956', 'label': 0}, {'image_id': 'ISIC_0009958', 'label': 0}, {'image_id': 'ISIC_0009959', 'label': 1}, {'image_id': 'ISIC_0009965', 'label': 0}, {'image_id': 'ISIC_0009970', 'label': 0}, {'image_id': 'ISIC_0009977', 'label': 0}, {'image_id': 'ISIC_0009980', 'label': 0}, {'image_id': 'ISIC_0009982', 'label': 0}, {'image_id': 'ISIC_0009988', 'label': 0}, {'image_id': 'ISIC_0009990', 'label': 0}, {'image_id': 'ISIC_0009992', 'label': 1}, {'image_id': 'ISIC_0009993', 'label': 1}, {'image_id': 'ISIC_0009994', 'label': 0}, {'image_id': 'ISIC_0009998', 'label': 0}, {'image_id': 'ISIC_0010004', 'label': 0}, {'image_id': 'ISIC_0010009', 'label': 0}, {'image_id': 'ISIC_0010011', 'label': 1}, {'image_id': 'ISIC_0010013', 'label': 0}, {'image_id': 'ISIC_0010016', 'label': 1}, {'image_id': 'ISIC_0010018', 'label': 0}, {'image_id': 'ISIC_0010020', 'label': 0}, {'image_id': 'ISIC_0010023', 'label': 1}, {'image_id': 'ISIC_0010028', 'label': 0}, {'image_id': 'ISIC_0010033', 'label': 0}, {'image_id': 'ISIC_0010034', 'label': 1}, {'image_id': 'ISIC_0010037', 'label': 0}, {'image_id': 'ISIC_0010038', 'label': 0}, {'image_id': 'ISIC_0010041', 'label': 0}, {'image_id': 'ISIC_0010047', 'label': 0}, {'image_id': 'ISIC_0010055', 'label': 0}, {'image_id': 'ISIC_0010058', 'label': 0}, {'image_id': 'ISIC_0010059', 'label': 0}, {'image_id': 'ISIC_0010061', 'label': 0}, {'image_id': 'ISIC_0010062', 'label': 0}, {'image_id': 'ISIC_0010073', 'label': 0}, {'image_id': 'ISIC_0010077', 'label': 0}, {'image_id': 'ISIC_0010088', 'label': 0}, {'image_id': 'ISIC_0010089', 'label': 1}, {'image_id': 'ISIC_0010092', 'label': 0}, {'image_id': 'ISIC_0010100', 'label': 0}, {'image_id': 'ISIC_0010103', 'label': 0}, {'image_id': 'ISIC_0010171', 'label': 1}, {'image_id': 'ISIC_0010173', 'label': 0}, {'image_id': 'ISIC_0010175', 'label': 1}, {'image_id': 'ISIC_0010180', 'label': 0}, {'image_id': 'ISIC_0010183', 'label': 0}, {'image_id': 'ISIC_0010187', 'label': 0}, {'image_id': 'ISIC_0010190', 'label': 0}, {'image_id': 'ISIC_0010192', 'label': 0}, {'image_id': 'ISIC_0010193', 'label': 0}, {'image_id': 'ISIC_0010201', 'label': 0}, {'image_id': 'ISIC_0010202', 'label': 0}, {'image_id': 'ISIC_0010206', 'label': 0}, {'image_id': 'ISIC_0010207', 'label': 1}, {'image_id': 'ISIC_0010215', 'label': 0}, {'image_id': 'ISIC_0010216', 'label': 0}, {'image_id': 'ISIC_0010229', 'label': 0}, {'image_id': 'ISIC_0010231', 'label': 1}, {'image_id': 'ISIC_0010234', 'label': 0}, {'image_id': 'ISIC_0010238', 'label': 0}, {'image_id': 'ISIC_0010254', 'label': 0}, {'image_id': 'ISIC_0010255', 'label': 0}, {'image_id': 'ISIC_0010257', 'label': 0}, {'image_id': 'ISIC_0010261', 'label': 0}, {'image_id': 'ISIC_0010326', 'label': 0}, {'image_id': 'ISIC_0010331', 'label': 0}, {'image_id': 'ISIC_0010336', 'label': 0}, {'image_id': 'ISIC_0010340', 'label': 0}, {'image_id': 'ISIC_0010346', 'label': 0}, {'image_id': 'ISIC_0010347', 'label': 0}, {'image_id': 'ISIC_0010348', 'label': 0}, {'image_id': 'ISIC_0010360', 'label': 1}, {'image_id': 'ISIC_0010368', 'label': 0}, {'image_id': 'ISIC_0010369', 'label': 1}, {'image_id': 'ISIC_0010373', 'label': 0}, {'image_id': 'ISIC_0010374', 'label': 0}, {'image_id': 'ISIC_0010377', 'label': 0}, {'image_id': 'ISIC_0010378', 'label': 0}, {'image_id': 'ISIC_0010379', 'label': 1}, {'image_id': 'ISIC_0010437', 'label': 0}, {'image_id': 'ISIC_0010444', 'label': 0}, {'image_id': 'ISIC_0010448', 'label': 0}, {'image_id': 'ISIC_0010449', 'label': 0}, {'image_id': 'ISIC_0010451', 'label': 0}, {'image_id': 'ISIC_0010452', 'label': 0}, {'image_id': 'ISIC_0010454', 'label': 1}, {'image_id': 'ISIC_0010456', 'label': 0}, {'image_id': 'ISIC_0010460', 'label': 0}, {'image_id': 'ISIC_0010463', 'label': 0}, {'image_id': 'ISIC_0010474', 'label': 0}, {'image_id': 'ISIC_0010477', 'label': 0}, {'image_id': 'ISIC_0010483', 'label': 0}, {'image_id': 'ISIC_0010494', 'label': 0}, {'image_id': 'ISIC_0010498', 'label': 0}, {'image_id': 'ISIC_0010552', 'label': 0}, {'image_id': 'ISIC_0010553', 'label': 0}, {'image_id': 'ISIC_0010556', 'label': 0}, {'image_id': 'ISIC_0010565', 'label': 0}, {'image_id': 'ISIC_0010574', 'label': 0}, {'image_id': 'ISIC_0010582', 'label': 0}, {'image_id': 'ISIC_0010584', 'label': 0}, {'image_id': 'ISIC_0010587', 'label': 1}, {'image_id': 'ISIC_0010588', 'label': 0}, {'image_id': 'ISIC_0010591', 'label': 1}, {'image_id': 'ISIC_0010592', 'label': 0}, {'image_id': 'ISIC_0010596', 'label': 1}, {'image_id': 'ISIC_0010597', 'label': 1}, {'image_id': 'ISIC_0010598', 'label': 0}, {'image_id': 'ISIC_0010599', 'label': 0}, {'image_id': 'ISIC_0010604', 'label': 0}, {'image_id': 'ISIC_0010606', 'label': 0}, {'image_id': 'ISIC_0010607', 'label': 0}, {'image_id': 'ISIC_0010846', 'label': 0}, {'image_id': 'ISIC_0010847', 'label': 0}, {'image_id': 'ISIC_0010854', 'label': 0}, {'image_id': 'ISIC_0010856', 'label': 0}, {'image_id': 'ISIC_0010863', 'label': 1}, {'image_id': 'ISIC_0011080', 'label': 0}, {'image_id': 'ISIC_0011081', 'label': 0}, {'image_id': 'ISIC_0011083', 'label': 0}, {'image_id': 'ISIC_0011089', 'label': 0}, {'image_id': 'ISIC_0011090', 'label': 0}, {'image_id': 'ISIC_0011092', 'label': 0}, {'image_id': 'ISIC_0011098', 'label': 0}, {'image_id': 'ISIC_0011101', 'label': 0}, {'image_id': 'ISIC_0011104', 'label': 0}, {'image_id': 'ISIC_0011107', 'label': 0}, {'image_id': 'ISIC_0011110', 'label': 0}, {'image_id': 'ISIC_0011112', 'label': 1}, {'image_id': 'ISIC_0011122', 'label': 1}, {'image_id': 'ISIC_0011129', 'label': 0}, {'image_id': 'ISIC_0011132', 'label': 1}, {'image_id': 'ISIC_0011143', 'label': 0}, {'image_id': 'ISIC_0011148', 'label': 0}, {'image_id': 'ISIC_0011149', 'label': 0}, {'image_id': 'ISIC_0011150', 'label': 0}, {'image_id': 'ISIC_0011151', 'label': 1}, {'image_id': 'ISIC_0011152', 'label': 0}, {'image_id': 'ISIC_0011155', 'label': 0}, {'image_id': 'ISIC_0011162', 'label': 0}, {'image_id': 'ISIC_0011167', 'label': 0}, {'image_id': 'ISIC_0011168', 'label': 0}, {'image_id': 'ISIC_0011171', 'label': 0}, {'image_id': 'ISIC_0011172', 'label': 0}, {'image_id': 'ISIC_0011175', 'label': 0}, {'image_id': 'ISIC_0011176', 'label': 1}, {'image_id': 'ISIC_0011177', 'label': 0}, {'image_id': 'ISIC_0011204', 'label': 0}, {'image_id': 'ISIC_0011205', 'label': 0}, {'image_id': 'ISIC_0011213', 'label': 0}, {'image_id': 'ISIC_0011219', 'label': 0}, {'image_id': 'ISIC_0011224', 'label': 0}, {'image_id': 'ISIC_0011227', 'label': 0}, {'image_id': 'ISIC_0011292', 'label': 0}, {'image_id': 'ISIC_0011294', 'label': 0}, {'image_id': 'ISIC_0011298', 'label': 0}, {'image_id': 'ISIC_0011300', 'label': 0}, {'image_id': 'ISIC_0011305', 'label': 1}, {'image_id': 'ISIC_0011310', 'label': 0}, {'image_id': 'ISIC_0011319', 'label': 0}, {'image_id': 'ISIC_0011321', 'label': 0}, {'image_id': 'ISIC_0011325', 'label': 0}, {'image_id': 'ISIC_0011333', 'label': 0}, {'image_id': 'ISIC_0011336', 'label': 0}, {'image_id': 'ISIC_0011338', 'label': 0}, {'image_id': 'ISIC_0011344', 'label': 1}, {'image_id': 'ISIC_0011349', 'label': 1}, {'image_id': 'ISIC_0011359', 'label': 0}, {'image_id': 'ISIC_0011363', 'label': 0}, {'image_id': 'ISIC_0011367', 'label': 0}, {'image_id': 'ISIC_0011374', 'label': 1}, {'image_id': 'ISIC_0011384', 'label': 0}, {'image_id': 'ISIC_0011386', 'label': 0}, {'image_id': 'ISIC_0011392', 'label': 0}]

MODULES = {'attention_pipeline_core.py': '"""Inference and Gradio UI for the frozen Attention U-Net melanoma candidate.\n\nNo training or threshold selection occurs in this module. JPEG ROI encoding is\npreserved because the training notebook classified saved quality-95 JPEG ROIs.\n"""\nfrom __future__ import annotations\n\nfrom io import BytesIO\nfrom pathlib import Path\nimport hashlib\nimport json\nimport threading\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom scipy import ndimage\nimport tensorflow as tf\n\nSEG_SIZE = (256, 256)\nCLS_SIZE = (224, 224)\n\n\ndef image_tensor(path, size, normalise=False):\n    pixels = tf.io.decode_image(tf.io.read_file(str(path)), channels=3,\n                                expand_animations=False)\n    pixels = tf.cast(tf.image.resize(pixels, size, method="bilinear"), tf.float32)\n    return pixels / 255.0 if normalise else pixels\n\n\ndef roi_tensor(roi):\n    # Match roi.save(..., quality=95) -> decode_classifier_image() in training.\n    buffer = BytesIO()\n    roi.convert("RGB").save(buffer, format="JPEG", quality=95)\n    pixels = tf.io.decode_jpeg(buffer.getvalue(), channels=3)\n    return tf.cast(tf.image.resize(pixels, CLS_SIZE), tf.float32)\n\n\ndef crop_from_prediction(image_pil, predicted_mask, padding=0.15):\n    image = image_pil.convert("RGB")\n    width, height = image.size\n    binary = Image.fromarray(np.uint8(np.asarray(predicted_mask) > 0) * 255)\n    binary = np.asarray(binary.resize(image.size, Image.Resampling.NEAREST)) > 0\n    labelled, count = ndimage.label(binary)\n    reason = "cropped"\n    if count == 0:\n        crop, reason = image, "empty predicted mask"\n    else:\n        sizes = np.bincount(labelled.ravel())\n        sizes[0] = 0\n        lesion = labelled == int(sizes.argmax())\n        area = int(lesion.sum())\n        if area < max(64, int(width * height * 0.001)) or area > width * height * 0.80:\n            crop, reason = image, "implausible predicted area"\n        else:\n            ys, xs = np.where(lesion)\n            span = max(np.ptp(xs) + 1, np.ptp(ys) + 1)\n            margin = int(np.ceil(span * padding))\n            x0 = max(0, int(xs.min()) - margin)\n            x1 = min(width, int(xs.max()) + margin + 1)\n            y0 = max(0, int(ys.min()) - margin)\n            y1 = min(height, int(ys.max()) + margin + 1)\n            crop = image.crop((x0, y0, x1, y1))\n    a = np.asarray(crop)\n    h, w = a.shape[:2]\n    side = max(h, w)\n    top, left = (side - h) // 2, (side - w) // 2\n    square = np.pad(a, ((top, side - h - top), (left, side - w - left), (0, 0)),\n                    mode="edge")\n    roi = Image.fromarray(square).resize((CLS_SIZE[1], CLS_SIZE[0]),\n                                         Image.Resampling.BILINEAR)\n    return roi, reason\n\n\ndef mask_overlay(image, mask):\n    base = np.asarray(image.convert("RGB"), dtype=np.float32)\n    mask = Image.fromarray(np.uint8(mask) * 255).resize(\n        image.size, Image.Resampling.NEAREST)\n    binary = np.asarray(mask) > 127\n    tint = np.array([35, 195, 145], dtype=np.float32)\n    base[binary] = 0.65 * base[binary] + 0.35 * tint\n    boundary = binary & ~ndimage.binary_erosion(binary)\n    base[boundary] = [255, 215, 30]\n    return Image.fromarray(np.uint8(np.clip(base, 0, 255)))\n\n\ndef gradcam_overlay(roi, heatmap):\n    from matplotlib import colormaps\n    image = roi.convert("RGB").resize((CLS_SIZE[1], CLS_SIZE[0]))\n    heat = Image.fromarray(np.uint8(np.clip(heatmap, 0, 1) * 255)).resize(\n        image.size, Image.Resampling.BILINEAR)\n    colors = colormaps["jet"](np.asarray(heat, dtype=np.float32) / 255.0)[..., :3]\n    base = np.asarray(image, dtype=np.float32) / 255.0\n    return Image.fromarray(np.uint8(255 * np.clip(0.62 * base + 0.38 * colors, 0, 1)))\n\n\nclass MelanomaPipeline:\n    def __init__(self, segmenter, classifier, configuration, run_dir=None):\n        self.segmenter = segmenter\n        self.classifier = classifier\n        self.configuration = dict(configuration)\n        self.run_dir = Path(run_dir) if run_dir else None\n        self.lock = threading.RLock()\n        cfg = self.configuration\n        if cfg.get("view") != "dual":\n            raise ValueError("This app requires the full-image + ROI classifier (view=\'dual\').")\n        for field in ("threshold", "segmentation_threshold"):\n            if field not in cfg or not 0 <= float(cfg[field]) <= 1:\n                raise ValueError(f"Missing or invalid saved configuration field: {field}")\n        for field in ("method", "segmentation_method"):\n            if cfg.get(field) not in ("single", "4-flip average"):\n                raise ValueError(f"Unsupported inference method: {cfg.get(field)}")\n        if segmenter.input_shape[1:] != (*SEG_SIZE, 3):\n            raise ValueError("Segmenter input must be 256 x 256 RGB.")\n        if len(classifier.inputs) != 2:\n            raise ValueError("Expected two classifier inputs: full image and ROI.")\n        shapes = [tuple(x.shape[1:]) for x in classifier.inputs]\n        if shapes != [(*CLS_SIZE, 3), (*CLS_SIZE, 3)]:\n            raise ValueError(f"Classifier input shapes are incompatible: {shapes}")\n\n    @classmethod\n    def load(cls, run_dir):\n        run_dir = Path(run_dir)\n        paths = {\n            "configuration": run_dir / "tables" / "selected_configuration.json",\n            "segmenter": run_dir / "models" / "attention_unet_best.keras",\n            "classifier": run_dir / "models" / "selected_final_classifier.keras",\n        }\n        missing = [str(p) for p in paths.values() if not p.is_file()]\n        if missing:\n            raise FileNotFoundError("The saved pipeline is incomplete. Missing:\\n" + "\\n".join(missing))\n        configuration = json.loads(paths["configuration"].read_text())\n        provenance = run_dir / "tables" / "segmentation_configuration.json"\n        if provenance.is_file():\n            expected = json.loads(provenance.read_text()).get("checkpoint_sha256")\n            if expected:\n                actual = hashlib.sha256(paths["segmenter"].read_bytes()).hexdigest()\n                if actual != expected:\n                    raise ValueError("The segmentation checkpoint does not match its saved configuration.")\n        manifest_path = run_dir / "deployment_manifest.json"\n        if manifest_path.is_file():\n            for relative, expected in json.loads(manifest_path.read_text()).get("sha256", {}).items():\n                actual = hashlib.sha256((run_dir / relative).read_bytes()).hexdigest()\n                if actual != expected:\n                    raise ValueError(f"Deployment bundle file has changed: {relative}")\n        # compile=False avoids needing custom training losses for inference.\n        segmenter = tf.keras.models.load_model(paths["segmenter"], compile=False)\n        classifier = tf.keras.models.load_model(paths["classifier"], compile=False)\n        return cls(segmenter, classifier, configuration, run_dir)\n\n    def segmentation_probabilities(self, images, batch_size=4):\n        images = np.asarray(images, dtype=np.float32)\n        methods = [(), (2,), (1,), (1, 2)] if self.configuration[\n            "segmentation_method"] == "4-flip average" else [()]\n        predictions = []\n        for axes in methods:\n            transformed = np.flip(images, axis=axes) if axes else images\n            score = self.segmenter.predict(np.ascontiguousarray(transformed),\n                                            batch_size=batch_size, verbose=0)[..., 0]\n            score = np.flip(score, axis=axes) if axes else score\n            predictions.append(score)\n        return np.mean(np.stack(predictions), axis=0)\n\n    def classifier_probabilities(self, full_images, roi_images, batch_size=8):\n        full_images = np.asarray(full_images, dtype=np.float32)\n        roi_images = np.asarray(roi_images, dtype=np.float32)\n        methods = [(), (2,), (1,), (1, 2)] if self.configuration[\n            "method"] == "4-flip average" else [()]\n        predictions = []\n        for axes in methods:\n            full = np.flip(full_images, axis=axes) if axes else full_images\n            roi = np.flip(roi_images, axis=axes) if axes else roi_images\n            features = {"full_image": np.ascontiguousarray(full),\n                        "roi_image": np.ascontiguousarray(roi)}\n            predictions.append(self.classifier.predict(\n                features, batch_size=batch_size, verbose=0).ravel())\n        return np.mean(np.stack(predictions), axis=0)\n\n    def gradcam(self, full_batch, roi_batch, target_class):\n        model = self.classifier\n        backbone = model.get_layer("efficientnetb0")\n        gap = model.get_layer("gap")\n        with tf.GradientTape() as tape:\n            full_map = backbone(full_batch, training=False)\n            roi_map = backbone(roi_batch, training=False)\n            tape.watch(roi_map)\n            features = model.get_layer("combine_features")([gap(full_map), gap(roi_map)])\n            hidden = model.get_layer("hidden_dense")(features)\n            hidden = model.get_layer("head_dropout")(hidden, training=False)\n            output = model.get_layer("prediction")\n            logit = tf.linalg.matmul(hidden, output.kernel) + output.bias\n            score = logit[:, 0] if target_class == 1 else -logit[:, 0]\n        gradients = tape.gradient(score, roi_map)\n        if gradients is None:\n            raise RuntimeError("Grad-CAM could not trace the ROI feature map.")\n        weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)\n        heatmap = tf.nn.relu(tf.reduce_sum(roi_map * weights, axis=-1)[0])\n        return (heatmap / (tf.reduce_max(heatmap) + 1e-8)).numpy()\n\n    def predict_file(self, path, explain=True):\n        if path is None:\n            raise ValueError("Upload a dermoscopic lesion image first.")\n        path = Path(path)\n        with Image.open(path) as handle:\n            original = handle.convert("RGB")\n        with self.lock:\n            segmentation_input = image_tensor(path, SEG_SIZE, normalise=True)[None, ...]\n            probabilities = self.segmentation_probabilities(segmentation_input)[0]\n            mask = probabilities >= float(self.configuration["segmentation_threshold"])\n            roi, reason = crop_from_prediction(\n                original, mask, padding=float(self.configuration.get("roi_padding", 0.15)))\n            full_input = image_tensor(path, CLS_SIZE)[None, ...]\n            roi_input = roi_tensor(roi)[None, ...]\n            score = float(self.classifier_probabilities(full_input, roi_input)[0])\n            predicted = int(score >= float(self.configuration["threshold"]))\n            heatmap = self.gradcam(full_input, roi_input, predicted) if explain else None\n        result = {\n            "original": original, "mask": Image.fromarray(np.uint8(mask) * 255),\n            "mask_overlay": mask_overlay(original, mask), "roi": roi,\n            "score": score, "predicted_label": predicted,\n            "label": "Melanoma" if predicted else "Benign",\n            "roi_reason": reason, "probability_map": probabilities,\n        }\n        result["gradcam"] = gradcam_overlay(roi, heatmap) if heatmap is not None else None\n        return result\n\n\ndef build_demo(pipeline, sample_paths=()):\n    import gradio as gr\n    cfg = pipeline.configuration\n\n    def preview(path):\n        if path is None:\n            return None\n        with Image.open(path) as image:\n            return image.convert("RGB")\n\n    def analyse(path):\n        try:\n            result = pipeline.predict_file(path)\n        except (ValueError, FileNotFoundError, OSError) as error:\n            raise gr.Error(str(error)) from error\n        summary = f"### Model prediction: **{result[\'label\']}**\\n"\n        summary += (f"Melanoma model score: **{result[\'score\']:.4f}**. "\n                    f"Decision threshold: {float(cfg[\'threshold\']):.4f}.\\n\\n"\n                    "The score is a model output, not a calibrated clinical probability.")\n        if result["roi_reason"] != "cropped":\n            summary += "\\n\\nThe lesion crop was unreliable; the classifier used the full image for the ROI view."\n        panels = [(result["original"], "Original image"),\n                  (result["mask"], "Predicted lesion mask"),\n                  (result["mask_overlay"], "Lesion outline"),\n                  (result["roi"], "Lesion ROI"),\n                  (result["gradcam"], "Grad-CAM on ROI")]\n        return summary, panels\n\n    def clear():\n        return None, None, "Upload a dermoscopic lesion image, then select Analyse.", []\n\n    with gr.Blocks(title="Attention U-Net melanoma research demo",\n                   delete_cache=(3600, 3600)) as demo:\n        gr.Markdown("# Melanoma image analysis\\n"\n                    "Attention U-Net lesion segmentation with EfficientNetB0 classification.\\n\\n"\n                    "**Student research prototype. Not validated for clinical diagnosis.**")\n        with gr.Tab("Analyse image"):\n            with gr.Row():\n                with gr.Column(scale=1):\n                    upload = gr.File(label="Upload dermoscopic image (JPG or PNG)",\n                                     file_types=[".jpg", ".jpeg", ".png"], type="filepath")\n                    input_preview = gr.Image(label="Uploaded image", interactive=False)\n                    with gr.Row():\n                        analyse_button = gr.Button("Analyse", variant="primary")\n                        clear_button = gr.Button("Clear")\n                    if sample_paths:\n                        gr.Examples(examples=[[str(p)] for p in sample_paths],\n                                    inputs=upload, label="Public ISIC example images")\n                with gr.Column(scale=2):\n                    summary = gr.Markdown("Upload a dermoscopic lesion image, then select Analyse.")\n                    panels = gr.Gallery(label="Pipeline outputs", columns=3,\n                                        object_fit="contain", height="auto")\n                    gr.Markdown("The lesion mask estimates the lesion boundary. "\n                                "Grad-CAM shows image regions contributing to the model prediction; "\n                                "it is not a lesion mask.")\n            upload.change(preview, inputs=upload, outputs=input_preview)\n            analyse_button.click(analyse, inputs=upload, outputs=[summary, panels],\n                                 api_name="analyse", concurrency_limit=1)\n            clear_button.click(clear, outputs=[upload, input_preview, summary, panels])\n        with gr.Tab("Evaluation"):\n            gr.Markdown("## Saved evaluation results\\n"\n                        "Classification and segmentation measure different tasks. "\n                        "The official ISIC test set was viewed in earlier experiments; "\n                        "these results are exploratory follow-up.")\n            run_dir = pipeline.run_dir\n            displayed = False\n            if run_dir:\n                for filename, label in [("complete_segmentation_test_metrics.csv", "Lesion segmentation"),\n                                        ("complete_classification_test_metrics.csv", "Melanoma classification")]:\n                    path = run_dir / "tables" / filename\n                    if path.is_file():\n                        gr.Markdown("### " + label)\n                        gr.Dataframe(value=pd.read_csv(path).T.reset_index(),\n                                     headers=["Metric", "Value"], interactive=False)\n                        displayed = True\n                figures = sorted((run_dir / "figures").glob("*.png"))\n                if figures:\n                    gr.Gallery(value=[(str(p), p.stem.replace("_", " ")) for p in figures],\n                               label="Evaluation figures", columns=1, object_fit="contain")\n                    displayed = True\n            if not displayed:\n                gr.Markdown("Run the reporting notebook to add test loss, metrics and evaluation figures.")\n    return demo\n', 'bestcase_comparison_helpers.py': 'from pathlib import Path\nfrom io import BytesIO\nimport gc\nimport json\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport tensorflow as tf\nimport cv2\nfrom attention_pipeline_core import MelanomaPipeline\nIMG_SIZE = 256\nCLS_IMG_SIZE = 224\nMASK_MODE = \'mask_crop\'\nSEG_IMAGE_SIZE = (256, 256)\nSEG_THRESHOLD = 0.5\n\ndef get_lesion_region(image, unet_model, pad_frac=0.15, mode=MASK_MODE):\n    """Runs the trained U-Net on a full-res image to get a lesion mask, then applies\n    pixel-masking and/or cropping (per `mode`) before resizing for the classifier."""\n    h, w = (image.shape[0], image.shape[1])\n    seg_input = tf.image.resize(image, [IMG_SIZE, IMG_SIZE]) / 255.0\n    seg_input = tf.expand_dims(seg_input, 0)\n    pred_mask_small = unet_model(seg_input, training=False)[0, ..., 0].numpy()\n    pred_mask_small = (pred_mask_small > 0.5).astype(np.uint8)\n    pred_mask_full = tf.image.resize(pred_mask_small[..., np.newaxis], [h, w], method=\'nearest\').numpy()[..., 0].astype(np.uint8)\n    ys, xs = np.where(pred_mask_small > 0)\n    lesion_found = len(xs) > 0 and len(ys) > 0\n    work_image = image.numpy() if hasattr(image, \'numpy\') else np.array(image)\n    if mode in (\'mask\', \'mask_crop\'):\n        if lesion_found:\n            work_image = work_image * pred_mask_full[..., np.newaxis]\n    if mode in (\'crop\', \'mask_crop\') and lesion_found:\n        x_min, x_max = (xs.min() / IMG_SIZE, xs.max() / IMG_SIZE)\n        y_min, y_max = (ys.min() / IMG_SIZE, ys.max() / IMG_SIZE)\n        pad_x = (x_max - x_min) * pad_frac\n        pad_y = (y_max - y_min) * pad_frac\n        x_min = max(0.0, x_min - pad_x)\n        x_max = min(1.0, x_max + pad_x)\n        y_min = max(0.0, y_min - pad_y)\n        y_max = min(1.0, y_max + pad_y)\n        x1, x2 = (int(x_min * w), int(x_max * w))\n        y1, y2 = (int(y_min * h), int(y_max * h))\n        work_image = work_image[y1:max(y2, y1 + 1), x1:max(x2, x1 + 1), :]\n    out = tf.image.resize(work_image, [CLS_IMG_SIZE, CLS_IMG_SIZE])\n    return out.numpy()\n\ndef predict_mask_for_pil(model, image_pil):\n    resized = image_pil.resize((SEG_IMAGE_SIZE[1], SEG_IMAGE_SIZE[0]), Image.Resampling.BILINEAR)\n    x = np.asarray(resized, dtype=np.float32) / 255.0\n    probability = model.predict(x[None, ...], verbose=0)[0, ..., 0]\n    binary = (probability >= SEG_THRESHOLD).astype(np.uint8)\n    return (probability, binary)\n\ndef extract_roi_from_binary_mask(image_pil, binary_mask, padding_fraction=0.1):\n    image_array = np.asarray(image_pil.convert(\'RGB\'))\n    h, w = image_array.shape[:2]\n    mask = np.asarray(Image.fromarray((binary_mask * 255).astype(np.uint8)).resize((w, h), Image.Resampling.NEAREST)) > 0\n    ys, xs = np.where(mask)\n    if len(xs) == 0 or len(ys) == 0:\n        return (image_pil.copy(), (0, 0, w, h), True)\n    x_min, x_max = (int(xs.min()), int(xs.max()))\n    y_min, y_max = (int(ys.min()), int(ys.max()))\n    box_w, box_h = (max(1, x_max - x_min + 1), max(1, y_max - y_min + 1))\n    pad_x = int(round(box_w * padding_fraction))\n    pad_y = int(round(box_h * padding_fraction))\n    x0, y0 = (max(0, x_min - pad_x), max(0, y_min - pad_y))\n    x1, y1 = (min(w, x_max + pad_x + 1), min(h, y_max + pad_y + 1))\n    if x1 - x0 < 8 or y1 - y0 < 8:\n        return (image_pil.copy(), (0, 0, w, h), True)\n    return (image_pil.crop((x0, y0, x1, y1)), (x0, y0, x1, y1), False)\n\ndef hard_overlap(truth, prediction):\n    truth = np.asarray(truth, dtype=bool)\n    prediction = np.asarray(prediction, dtype=bool)\n    if truth.shape != prediction.shape:\n        raise ValueError(f\'Mask shapes differ: {truth.shape} and {prediction.shape}\')\n    intersection = int(np.logical_and(truth, prediction).sum())\n    total = int(truth.sum()) + int(prediction.sum())\n    union = int(np.logical_or(truth, prediction).sum())\n    return {\'dice\': 2.0 * intersection / total if total else 1.0, \'iou\': intersection / union if union else 1.0}\n\ndef load_pair(segmenter_path, classifier_path):\n    seg = tf.keras.models.load_model(str(segmenter_path), compile=False)\n    cls = tf.keras.models.load_model(str(classifier_path), compile=False)\n    if tuple(seg.input_shape[1:]) != (256, 256, 3):\n        raise ValueError(\'Expected a 256 x 256 RGB segmenter.\')\n    if len(cls.inputs) != 1 or tuple(cls.input_shape[1:]) != (224, 224, 3):\n        raise ValueError(\'Expected a single-input 224 x 224 RGB classifier.\')\n    return (seg, cls)\n\ndef check_score(score):\n    score = float(score)\n    if not np.isfinite(score) or not 0 <= score <= 1:\n        raise ValueError(f\'Invalid sigmoid model score: {score}\')\n    return score\n', 'full_stage_visuals.py': '"""Export real inference stages from four existing melanoma pipelines. No training."""\nfrom pathlib import Path\nfrom io import BytesIO\nimport gc\nimport json\nimport zipfile\nfrom datetime import datetime, timezone\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport tensorflow as tf\nimport cv2\nimport matplotlib.pyplot as plt\nfrom scipy import ndimage\n\nfrom attention_pipeline_core import MelanomaPipeline, image_tensor, roi_tensor\nfrom bestcase_comparison_helpers import (\n    load_pair, get_lesion_region, predict_mask_for_pil,\n    extract_roi_from_binary_mask, hard_overlap, check_score,\n)\n\nORDER = [\'Attention U-Net\', \'U-Net (uploaded notebook)\', \'ResUNet\', \'TransUNet\']\nSLUGS = dict(zip(ORDER, [\'attention_unet\', \'unet\', \'resunet\', \'transunet\']))\nTITLES = [\'Original image\', \'Ground-truth mask\', \'Segmentation probability\',\n          \'Predicted lesion mask\', \'Mask overlay\', \'Isolated lesion\',\n          \'Classifier lesion ROI\', \'Grad-CAM heatmap\', \'Grad-CAM on ROI\']\nKEYS = [\'original\', \'ground_truth\', \'probability\', \'mask\', \'overlay\',\n        \'isolated\', \'roi\', \'heatmap\', \'gradcam\']\n\n\ndef array_rgb(pixels):\n    return np.uint8(np.clip(np.asarray(pixels), 0, 255))\n\n\ndef signed_logit_gradcam(model, image_batch, target_class=1):\n    """Follow each saved classifier\'s augmentation, backbone and head exactly.\n\n    All panels explain the melanoma logit, giving the same target across models.\n    Explicit input watching also supports fully frozen EfficientNet backbones.\n    """\n    candidates = [layer for layer in model.layers\n                  if isinstance(layer, tf.keras.Model)\n                  and \'efficientnet\' in layer.name.lower()\n                  and len(layer.output_shape) == 4]\n    if len(candidates) != 1:\n        raise ValueError(f\'{model.name}: expected one nested spatial backbone; \'\n                         f\'found {[layer.name for layer in candidates]}\')\n    backbone = candidates[0]\n    index = model.layers.index(backbone)\n    final = model.layers[-1]\n    if not isinstance(final, tf.keras.layers.Dense) or final.units != 1:\n        raise ValueError(\'Expected the saved one-unit sigmoid classification head.\')\n    batch = tf.convert_to_tensor(image_batch, dtype=tf.float32)\n    with tf.GradientTape() as tape:\n        tape.watch(batch)\n        value = batch\n        for layer in model.layers[:index]:\n            if not isinstance(layer, tf.keras.layers.InputLayer):\n                value = layer(value, training=False)\n        feature = backbone(value, training=False)\n        tape.watch(feature)\n        value = feature\n        for layer in model.layers[index + 1:-1]:\n            value = layer(value, training=False)\n        logit = tf.linalg.matmul(value, final.kernel)\n        if final.use_bias:\n            logit = logit + final.bias\n        score = logit[:, 0] if target_class == 1 else -logit[:, 0]\n        probability = tf.math.sigmoid(logit)[0, 0]\n    gradient = tape.gradient(score, feature)\n    if gradient is None:\n        raise RuntimeError(f\'{model.name}: Grad-CAM gradient could not be traced.\')\n    pooled = tf.reduce_mean(gradient, axis=(1, 2), keepdims=True)\n    heat = tf.nn.relu(tf.reduce_sum(feature * pooled, axis=-1)[0])\n    heat = heat / (tf.reduce_max(heat) + 1e-8)\n    return heat.numpy(), float(probability.numpy())\n\n\ndef detailed_attention(image_path, spec):\n    pipeline = MelanomaPipeline.load(spec[\'run_dir\'])\n    result = pipeline.predict_file(image_path, explain=False)\n    full = image_tensor(image_path, (224, 224))[None, ...]\n    roi = roi_tensor(result[\'roi\'])[None, ...]\n    heat = pipeline.gradcam(full, roi, target_class=1)\n    return dict(probability=result[\'probability_map\'],\n                mask=np.asarray(result[\'mask\']) > 0,\n                roi_pixels=roi.numpy()[0], heatmap=heat,\n                score=check_score(result[\'score\']),\n                threshold=float(pipeline.configuration[\'threshold\']),\n                mask_threshold=float(pipeline.configuration[\'segmentation_threshold\']),\n                mask_method=pipeline.configuration[\'segmentation_method\'],\n                classifier_method=pipeline.configuration[\'method\'],\n                roi_note=result[\'roi_reason\'],\n                full_classifier_input=full.numpy()[0])\n\n\ndef detailed_peer(name, image_path, spec):\n    segmenter, classifier = load_pair(spec[\'segmenter\'], spec[\'classifier\'])\n    with Image.open(image_path) as handle:\n        original = handle.convert(\'RGB\')\n    if name.startswith(\'U-Net (\'):\n        rgb = tf.cast(tf.io.decode_jpeg(tf.io.read_file(str(image_path)), channels=3),\n                      tf.float32)\n        seg_input = tf.image.resize(rgb, (256, 256)) / 255.0\n        probability = segmenter(seg_input[None, ...], training=False)[0, ..., 0].numpy()\n        mask = probability > 0.5\n        roi_pixels = get_lesion_region(rgb, segmenter, pad_frac=0.15, mode=\'mask_crop\')\n        note = \'Background masking and 15% bounding-box crop\'\n    elif name == \'ResUNet\':\n        probability, mask = predict_mask_for_pil(segmenter, original)\n        roi, _, fallback = extract_roi_from_binary_mask(original, mask, padding_fraction=0.10)\n        buffer = BytesIO()\n        roi.save(buffer, format=\'JPEG\', quality=95)\n        pixels = tf.io.decode_jpeg(buffer.getvalue(), channels=3)\n        roi_pixels = tf.cast(tf.image.resize(pixels, (224, 224), method=\'bilinear\'),\n                             tf.float32).numpy()\n        note = \'Full-image fallback\' if fallback else \'10% RGB bounding-box crop, quality-95 JPEG\'\n    elif name == \'TransUNet\':\n        pixels = cv2.imread(str(image_path), cv2.IMREAD_COLOR)\n        if pixels is None:\n            raise ValueError(\'OpenCV could not decode the original image.\')\n        pixels = cv2.cvtColor(pixels, cv2.COLOR_BGR2RGB)\n        seg_pixels = cv2.resize(pixels, (256, 256)).astype(np.float32) / 255.0\n        probability = segmenter.predict(seg_pixels[None, ...], verbose=0)[0, ..., 0]\n        mask = probability >= 0.5\n        masked = seg_pixels * mask[..., None].astype(np.float32)\n        roi_pixels = tf.image.resize(masked, (224, 224)).numpy() * 255.0\n        note = \'Background-masked 256-pixel full frame resized to 224\'\n    else:\n        raise ValueError(name)\n    batch = np.asarray(roi_pixels, dtype=np.float32)[None, ...]\n    score = check_score(classifier.predict(batch, verbose=0).ravel()[0])\n    heat, reconstructed_score = signed_logit_gradcam(classifier, batch, target_class=1)\n    if not np.isclose(score, reconstructed_score, atol=2e-5, rtol=2e-5):\n        raise RuntimeError(f\'{name}: Grad-CAM head does not reproduce the saved \'\n                           f\'classifier score ({reconstructed_score} vs {score}).\')\n    return dict(probability=probability, mask=np.asarray(mask, dtype=bool),\n                roi_pixels=roi_pixels, heatmap=heat, score=score, threshold=0.5,\n                mask_threshold=0.5, mask_method=\'single\', classifier_method=\'single\',\n                roi_note=note, full_classifier_input=None)\n\n\ndef stage_panels(original, truth, result):\n    rgb = np.asarray(original.resize((256, 256), Image.Resampling.BILINEAR))\n    mask = np.asarray(result[\'mask\'], dtype=bool)\n    if mask.shape != (256, 256) or np.shape(result[\'probability\']) != (256, 256):\n        raise ValueError(\'Expected a 256 x 256 segmentation map.\')\n    overlay = rgb.astype(np.float32).copy()\n    overlay[mask] = .65 * overlay[mask] + .35 * np.array([35, 195, 145])\n    border = mask & ~ndimage.binary_erosion(mask)\n    overlay[border] = [255, 215, 30]\n    roi = array_rgb(result[\'roi_pixels\'])\n    raw_heat = np.asarray(result[\'heatmap\'], dtype=np.float32)\n    if not np.isfinite(raw_heat).all() or raw_heat.ndim != 2:\n        raise ValueError(\'Grad-CAM must be a finite spatial map.\')\n    heat = tf.image.resize(raw_heat[None, ..., None], roi.shape[:2],\n                           method=\'bilinear\')[0, ..., 0].numpy()\n    heat = np.clip(heat, 0, 1)\n    colors = plt.colormaps[\'jet\'](heat)[..., :3]\n    cam = .62 * roi.astype(np.float32) + .38 * colors * 255\n    # Do not paint an apparent focus when every positive contribution is zero.\n    if float(raw_heat.max()) <= 1e-8:\n        cam = roi.copy()\n    return dict(original=rgb, ground_truth=np.uint8(truth) * 255,\n                probability=np.asarray(result[\'probability\'], dtype=np.float32),\n                mask=np.uint8(mask) * 255, overlay=array_rgb(overlay),\n                isolated=rgb * mask[..., None], roi=roi,\n                heatmap=heat, gradcam=array_rgb(cam))\n\n\ndef plot_grid(results, output, filename, keys=KEYS):\n    columns = len(keys)\n    width = 3.0 * columns + 2.7\n    fig, axes = plt.subplots(len(results), columns, figsize=(width, 2.9 * len(results) + 1.9),\n                             squeeze=False, facecolor=\'white\')\n    for row, result in enumerate(results):\n        for col, key in enumerate(keys):\n            ax = axes[row, col]\n            if key in (\'probability\', \'heatmap\'):\n                ax.imshow(result[\'panels\'][key], cmap=\'viridis\' if key == \'probability\'\n                          else \'jet\', vmin=0, vmax=1)\n            elif key in (\'ground_truth\', \'mask\'):\n                ax.imshow(result[\'panels\'][key], cmap=\'gray\', vmin=0, vmax=255)\n            else:\n                ax.imshow(result[\'panels\'][key])\n            ax.axis(\'off\')\n            if row == 0:\n                title = TITLES[KEYS.index(key)]\n                ax.set_title(title, fontsize=13, pad=14, fontweight=\'bold\')\n            if key in (\'heatmap\', \'gradcam\') and result[\'no_positive_cam\']:\n                ax.text(.5, -.035, \'No positive activation\', ha=\'center\', va=\'top\',\n                        transform=ax.transAxes, fontsize=10)\n        name = \'U-Net*\' if result[\'name\'].startswith(\'U-Net (\') else result[\'name\']\n        label = (f\'{name}\\nDice {result["dice"]:.4f}\\nIoU {result["iou"]:.4f}\\n\\n\'\n                 f\'Prediction: {result["prediction"]}\\nScore {result["score"]:.4f}\\n\'\n                 f\'Cutoff {result["threshold"]:.4f}\')\n        axes[row, 0].text(-.075, .5, label, transform=axes[row, 0].transAxes,\n                         ha=\'right\', va=\'center\', fontsize=11, linespacing=1.4)\n    image_id = results[0][\'image_id\']\n    true_label = results[0][\'true_label\']\n    fig.suptitle(f\'{image_id} | Actual label: {true_label} | All pipeline stages\',\n                 fontsize=22, fontweight=\'bold\', y=.985)\n    fig.text(.52, .035, \'Same selected image in every row. Grad-CAM target: melanoma class \'\n             \'(Attention: ROI branch). Maps are individually normalised; colours are not comparable in magnitude.\',\n             ha=\'center\', fontsize=11)\n    fig.text(.52, .012, results[0][\'selection_note\'] + \'; different training splits and ROI methods. \'\n             \'*The uploaded U-Net++ notebook implements U-Net. This is not an overall model ranking.\',\n             ha=\'center\', fontsize=11)\n    fig.subplots_adjust(left=2.5 / width, right=.992, top=.91, bottom=.08,\n                         hspace=.25, wspace=.08)\n    fig.savefig(output / filename, dpi=220, bbox_inches=\'tight\', facecolor=\'white\')\n    plt.close(fig)\n\n\ndef resolve_saved_models(drive_root, comparison_root, preferred_comparison=None):\n    """Resolve the checkpoints belonging to the completed comparison, without fitting."""\n    drive_root, comparison_root = Path(drive_root), Path(comparison_root)\n    preferred = Path(preferred_comparison) if preferred_comparison else None\n    manifests = ([preferred / \'comparison_provenance.json\'] if preferred else [])\n    manifests += sorted((comparison_root / \'comparisons\').glob(\'*/comparison_provenance.json\'),\n                        key=lambda p: p.parent.name, reverse=True)\n    manifests = [p for p in manifests if p.is_file()]\n    if not manifests:\n        raise FileNotFoundError(\'The completed comparison folder was not found. \'\n                                \'Use the same Drive account as your completed run.\')\n    manifest_path = manifests[0]\n    manifest = json.loads(manifest_path.read_text())\n    models = {\'Attention U-Net\': {\'run_dir\': Path(manifest[\'attention_saved_run\'])}}\n    originals = {\n        \'U-Net (uploaded notebook)\': (\n            drive_root / \'Melanoma_Comparison_Models/unet_segmenter.keras\',\n            drive_root / \'Melanoma_Comparison_Models/unet_classifier.keras\'),\n        \'ResUNet\': (\n            drive_root / \'Melanoma_AI_ResUNet_Targeted_Malignant_Aug_Results/models/resunet_best.keras\',\n            drive_root / \'Melanoma_AI_ResUNet_Targeted_Malignant_Aug_Results/models/efficientnet_b0_final.keras\'),\n        \'TransUNet\': (drive_root / \'43022/TransUNet_ISIC2016.keras\',\n                     drive_root / \'43022/EfficientNetB0_Augmented_Final.keras\'),\n    }\n    records = {row[\'Pipeline\']: row for row in manifest[\'peer_model_provenance\']}\n    missing = []\n    for name in ORDER[1:]:\n        if name not in records:\n            raise ValueError(\'Comparison provenance is missing \' + name)\n        record = records[name]\n        folder = comparison_root / \'trained_models\' / record[\'Code hash\'] / name.split()[0].lower()\n        cached = (folder / \'selected_segmenter.keras\', folder / \'selected_classifier.keras\')\n        if \'NEW\' in record[\'Weights\']:\n            pair = cached  # Do not silently substitute different weights.\n        else:\n            pair = cached if all(p.is_file() for p in cached) else originals[name]\n        missing.extend(str(p) for p in pair if not p.is_file())\n        models[name] = dict(segmenter=pair[0], classifier=pair[1])\n    attention = models[\'Attention U-Net\'][\'run_dir\']\n    missing.extend(str(attention / p) for p in (\n        \'models/attention_unet_best.keras\', \'models/selected_final_classifier.keras\',\n        \'tables/selected_configuration.json\') if not (attention / p).is_file())\n    if missing:\n        raise FileNotFoundError(\'Saved checkpoints are missing. No training was started.\\n\'\n                                + \'\\n\'.join(missing))\n    return models, manifest_path.parent, manifest\n\n\ndef export_all_stages(image_path, mask_path, models, destination, previous_comparison=None,\n                      source_manifest=None, true_label=None, selection_note=\'Selected illustrative case\'):\n    if true_label not in (\'Benign\', \'Melanoma\'):\n        raise ValueError(\'The verified actual label must be supplied explicitly.\')\n    image_id = Path(image_path).stem\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    with Image.open(image_path) as image:\n        original = image.convert(\'RGB\').copy()\n    with Image.open(mask_path) as mask:\n        truth = np.asarray(mask.convert(\'L\').resize((256, 256), Image.Resampling.NEAREST)) > 127\n    previous = None\n    if previous_comparison and (Path(previous_comparison) / \'single_image_comparison.csv\').is_file():\n        previous = pd.read_csv(Path(previous_comparison) / \'single_image_comparison.csv\')\n        if \'Image ID\' not in previous:\n            raise ValueError(\'The comparison CSV has no image identifier.\')\n        # Never compare the new melanoma case against the older benign case\'s scores.\n        previous = previous[previous[\'Image ID\'].astype(str) == image_id].set_index(\'Pipeline\')\n    results, rows = [], []\n    for name in ORDER:\n        tf.keras.backend.clear_session()\n        gc.collect()\n        print(\'Generating every stage:\', name, flush=True)\n        result = (detailed_attention(image_path, models[name]) if name == \'Attention U-Net\'\n                  else detailed_peer(name, image_path, models[name]))\n        result.update(name=name, image_id=image_id, true_label=true_label,\n                      selection_note=selection_note, **hard_overlap(truth, result[\'mask\']))\n        result[\'prediction\'] = \'Melanoma\' if result[\'score\'] >= result[\'threshold\'] else \'Benign\'\n        if previous is not None and name in previous.index:\n            for key, column in [(\'score\', \'Model score\'), (\'dice\', \'Hard Dice (this image)\'),\n                                (\'iou\', \'Hard IoU (this image)\'), (\'threshold\', \'Decision threshold\')]:\n                if not np.isclose(result[key], previous.loc[name, column], atol=5e-4, rtol=0):\n                    raise RuntimeError(f\'{name}: {key} differs from the completed comparison. \'\n                                       \'Check the checkpoint and inference configuration.\')\n        result[\'panels\'] = stage_panels(original, truth, result)\n        result[\'no_positive_cam\'] = float(np.max(result[\'heatmap\'])) <= 1e-8\n        folder = destination / \'individual_stages\' / SLUGS[name]\n        folder.mkdir(parents=True, exist_ok=True)\n        for key, value in result[\'panels\'].items():\n            if key in (\'probability\', \'heatmap\'):\n                np.save(folder / (key + \'.npy\'), value)\n                Image.fromarray(np.uint8(np.clip(value, 0, 1) * 255)).save(folder / (key + \'.png\'))\n            else:\n                Image.fromarray(value).save(folder / (key + \'.png\'))\n        np.save(folder / \'classifier_roi_input_float32.npy\', np.asarray(result[\'roi_pixels\'], np.float32))\n        np.save(folder / \'native_gradcam_heatmap.npy\', result[\'heatmap\'])\n        if result[\'full_classifier_input\'] is not None:\n            np.save(folder / \'classifier_full_input_float32.npy\', result[\'full_classifier_input\'])\n            Image.fromarray(array_rgb(result[\'full_classifier_input\'])).save(folder / \'classifier_full_input.png\')\n        rows.append({\'Pipeline\': name, \'Image ID\': image_id, \'True label\': true_label,\n                     \'Prediction\': result[\'prediction\'], \'Model score\': result[\'score\'],\n                     \'Correct classification\': result[\'prediction\'] == true_label,\n                     \'Decision threshold\': result[\'threshold\'], \'Dice\': result[\'dice\'],\n                     \'IoU\': result[\'iou\'], \'Mask inference\': result[\'mask_method\'],\n                     \'Classifier inference\': result[\'classifier_method\'],\n                     \'ROI treatment\': result[\'roi_note\'], \'Grad-CAM target\': \'Melanoma logit\',\n                     \'No positive Grad-CAM activation\': result[\'no_positive_cam\']})\n        results.append(result)\n        tf.keras.backend.clear_session()\n        gc.collect()\n    plot_grid(results, destination, \'FULL_COMPARISON_ALL_STAGES.png\')\n    plot_grid(results, destination, \'SLIDE_A_SEGMENTATION_COMPARISON.png\', keys=KEYS[:5])\n    plot_grid(results, destination, \'SLIDE_B_ROI_AND_GRADCAM_COMPARISON.png\', keys=KEYS[5:])\n    for result in results:\n        plot_grid([result], destination, SLUGS[result[\'name\']] + \'_ALL_STAGES.png\')\n    table = pd.DataFrame(rows)\n    table.to_csv(destination / \'all_stage_predictions.csv\', index=False)\n    manifest = {\'source_model_comparison\': dict(source_manifest or {})}\n    manifest.update(image_id=image_id, true_label=true_label,\n                    gradcam_target=\'Melanoma logit for every pipeline; Attention ROI branch\',\n                    heatmaps=\'Normalised separately within each image/model\',\n                    original_comparison=str(previous_comparison),\n                    checkpoint_paths={name: {k: str(v) for k, v in spec.items()}\n                                      for name, spec in models.items()},\n                    created_utc=datetime.now(timezone.utc).isoformat(),\n                    selection=selection_note)\n    (destination / \'full_stage_provenance.json\').write_text(json.dumps(manifest, indent=2))\n    (destination / \'READ_ME.txt\').write_text(\n        \'FULL_COMPARISON_ALL_STAGES.png contains every stage for every pipeline.\\n\'\n        \'SLIDE_A and SLIDE_B split the same figure into readable presentation panels.\\n\'\n        \'Each pipeline also has its own full-stage image and a folder of separate PNGs.\\n\'\n        \'Ground truth is the expert lesion mask, not another model prediction.\\n\'\n        \'Probability maps are the segmenter output before thresholding.\\n\'\n        \'Isolated lesion is a visualisation made by multiplying the original image by \'\n        \'the predicted mask; each model retains its original classifier ROI treatment.\\n\'\n        \'Classifier ROI panels show the actual inference input, including JPEG preprocessing \'\n        \'where applicable. Exact float32 tensors are included.\\n\'\n        \'Grad-CAM explains the melanoma class in all rows. It is not a lesion mask \'\n        \'or an indication of clinical certainty. Attention shows its ROI branch; \'\n        \'that classifier also uses the full image. Its full-image input is exported separately.\\n\'\n        \'Colours use independent normalisation. A zero map has no positive activation.\\n\'\n        f\'These figures concern selected case {image_id} (actual label: {true_label}), \'\n        \'not overall pipeline performance.\\n\'\n        \'Peer models were newly trained in the previous comparison; training splits differ.\\n\'\n        \'The supplied U-Net++ notebook implements standard U-Net.\\n\')\n    package = destination / \'Melanoma_FULL_STAGE_VISUAL_COMPARISON.zip\'\n    with zipfile.ZipFile(package, \'w\', zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(destination.rglob(\'*\')):\n            if path.is_file() and path != package:\n                archive.write(path, path.relative_to(destination))\n    return package, table\n', 'attention_case_search.py': '"""Find clearly labelled illustrative Attention advantages using saved pipelines."""\nfrom pathlib import Path\nfrom io import BytesIO\nimport gc\nimport hashlib\nimport json\nimport shutil\nimport zipfile\nimport urllib.request\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport tensorflow as tf\nimport cv2\n\nfrom attention_pipeline_core import (MelanomaPipeline, image_tensor, roi_tensor,\n                                     crop_from_prediction)\nfrom bestcase_comparison_helpers import (load_pair, get_lesion_region,\n                                        extract_roi_from_binary_mask, hard_overlap)\nfrom full_stage_visuals import ORDER, SLUGS, export_all_stages\n\nDATA_URL = (\'https://isic-archive.s3.amazonaws.com/challenges/2016/\'\n            \'ISBI2016_ISIC_Part3B_Test_Data.zip\')\n\n\ndef sha_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef prepare_cases(labels, dataset_dir, runtime_dir, scope=\'Melanoma only\'):\n    frame = labels.copy()\n    if scope == \'Melanoma only\':\n        frame = frame[frame[\'label\'] == 1]\n    elif scope != \'All test cases\':\n        raise ValueError(\'Choose Melanoma only or All test cases.\')\n    frame = frame.sort_values(\'image_id\').reset_index(drop=True)\n    dataset_dir, runtime_dir = Path(dataset_dir), Path(runtime_dir)\n    dataset_dir.mkdir(parents=True, exist_ok=True)\n    runtime_dir.mkdir(parents=True, exist_ok=True)\n    image_dir, mask_dir = dataset_dir / \'images\', dataset_dir / \'masks\'\n    image_dir.mkdir(exist_ok=True)\n    mask_dir.mkdir(exist_ok=True)\n    records = {}\n    for row in frame.itertuples():\n        records[row.image_id + \'.jpg\'] = image_dir / (row.image_id + \'.jpg\')\n        records[row.image_id + \'_Segmentation.png\'] = mask_dir / (row.image_id + \'_Segmentation.png\')\n    missing = {name: path for name, path in records.items() if not path.is_file()}\n    if missing:\n        # Reuse the original comparison runtime\'s extracted official test files.\n        original = Path(\'/content/melanoma_bestcase_comparison/isic2016\')\n        for name, path in list(missing.items()):\n            folder = (\'ISBI2016_ISIC_Part1_Test_Data\' if name.endswith(\'.jpg\')\n                      else \'ISBI2016_ISIC_Part1_Test_GroundTruth\')\n            source = original / folder / name\n            if source.is_file():\n                shutil.copyfile(source, path)\n                del missing[name]\n    if missing:\n        archive_path = runtime_dir / \'ISIC2016_official_test.zip\'\n        if not archive_path.is_file() or not zipfile.is_zipfile(archive_path):\n            print(\'Downloading the official test archive once (about 245 MB). No training.\', flush=True)\n            partial = archive_path.with_suffix(\'.partial\')\n            with urllib.request.urlopen(DATA_URL, timeout=180) as response, partial.open(\'wb\') as target:\n                shutil.copyfileobj(response, target, length=8 * 1024 * 1024)\n            if not zipfile.is_zipfile(partial):\n                raise ValueError(\'The downloaded test archive is incomplete. Rerun the cell.\')\n            partial.replace(archive_path)\n        with zipfile.ZipFile(archive_path) as archive:\n            members = {Path(name).name: name for name in archive.namelist() if not name.endswith(\'/\')}\n            for name, path in missing.items():\n                if name not in members:\n                    raise FileNotFoundError(\'Official reference file missing from archive: \' + name)\n                path.write_bytes(archive.read(members[name]))\n    frame[\'image_path\'] = frame[\'image_id\'].map(lambda name: str(image_dir / (name + \'.jpg\')))\n    frame[\'mask_path\'] = frame[\'image_id\'].map(lambda name: str(mask_dir / (name + \'_Segmentation.png\')))\n    if frame.empty:\n        raise ValueError(\'No reference cases were selected.\')\n    return frame\n\n\nclass FixedProbabilityModel:\n    """Reuse the predicted map inside the original U-Net ROI function unchanged."""\n    def __init__(self, probability):\n        self.output = tf.convert_to_tensor(np.asarray(probability)[None, ..., None], tf.float32)\n\n    def __call__(self, *args, **kwargs):\n        return self.output\n\n\ndef rgb_tensor(path):\n    return tf.cast(tf.io.decode_jpeg(tf.io.read_file(str(path)), channels=3), tf.float32)\n\n\ndef scan_fingerprint(name, spec, frame):\n    if name == \'Attention U-Net\':\n        run = Path(spec[\'run_dir\'])\n        paths = [run / \'models/attention_unet_best.keras\',\n                 run / \'models/selected_final_classifier.keras\',\n                 run / \'tables/selected_configuration.json\']\n    else:\n        paths = [Path(spec[\'segmenter\']), Path(spec[\'classifier\'])]\n    payload = dict(name=name, model_sha256=[sha_file(path) for path in paths],\n                   cases=frame[[\'image_id\', \'label\']].to_dict(\'records\'),\n                   scanner_sha256=sha_file(__file__))\n    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()\n\n\ndef scan_pipeline(name, spec, frame, cache_dir, batch_size=4):\n    cache_dir = Path(cache_dir)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    signature = scan_fingerprint(name, spec, frame)\n    csv_path = cache_dir / (SLUGS[name] + \'_\' + signature[:16] + \'.csv\')\n    if csv_path.is_file():\n        cached = pd.read_csv(csv_path)\n        if (len(cached) == len(frame)\n                and cached[\'Image ID\'].tolist() == frame[\'image_id\'].tolist()\n                and set(cached[\'Pipeline\']) == {name}):\n            print(\'Reusing completed search results:\', name, flush=True)\n            return cached\n    print(\'Scanning:\', name, \'| Cases:\', len(frame), flush=True)\n    tf.keras.backend.clear_session()\n    gc.collect()\n    pipeline = None\n    segmenter = classifier = None\n    rows = []\n    try:\n        if name == \'Attention U-Net\':\n            pipeline = MelanomaPipeline.load(spec[\'run_dir\'])\n            threshold = float(pipeline.configuration[\'threshold\'])\n        else:\n            segmenter, classifier = load_pair(spec[\'segmenter\'], spec[\'classifier\'])\n            threshold = .5\n        for start in range(0, len(frame), batch_size):\n            part = frame.iloc[start:start + batch_size]\n            case_rows = list(part.itertuples())\n            with_images = []\n            native_rgb = []\n            seg_images = []\n            for row in case_rows:\n                with Image.open(row.image_path) as handle:\n                    original = handle.convert(\'RGB\').copy()\n                with_images.append(original)\n                if name in (\'Attention U-Net\', \'U-Net (uploaded notebook)\'):\n                    rgb = rgb_tensor(row.image_path)\n                    native_rgb.append(rgb)\n                    seg_images.append((tf.image.resize(rgb, (256, 256)) / 255.0).numpy())\n                elif name == \'ResUNet\':\n                    seg_images.append(np.asarray(original.resize((256, 256), Image.Resampling.BILINEAR),\n                                                  dtype=np.float32) / 255.0)\n                else:\n                    pixels = cv2.imread(row.image_path, cv2.IMREAD_COLOR)\n                    if pixels is None:\n                        raise ValueError(\'Could not read image: \' + row.image_path)\n                    pixels = cv2.cvtColor(pixels, cv2.COLOR_BGR2RGB)\n                    seg_images.append(cv2.resize(pixels, (256, 256)).astype(np.float32) / 255.0)\n            seg_batch = np.stack(seg_images)\n            if pipeline is not None:\n                probabilities = pipeline.segmentation_probabilities(seg_batch, batch_size=batch_size)\n                masks = probabilities >= float(pipeline.configuration[\'segmentation_threshold\'])\n            else:\n                probabilities = segmenter.predict(seg_batch, batch_size=batch_size, verbose=0)[..., 0]\n                masks = (probabilities > .5 if name.startswith(\'U-Net (\') else probabilities >= .5)\n            rois, full = [], []\n            for index, row in enumerate(case_rows):\n                if pipeline is not None:\n                    roi, _ = crop_from_prediction(with_images[index], masks[index],\n                            padding=float(pipeline.configuration.get(\'roi_padding\', .15)))\n                    rois.append(roi_tensor(roi).numpy())\n                    full.append(tf.image.resize(native_rgb[index], (224, 224)).numpy())\n                elif name.startswith(\'U-Net (\'):\n                    rois.append(get_lesion_region(native_rgb[index],\n                                FixedProbabilityModel(probabilities[index]),\n                                pad_frac=.15, mode=\'mask_crop\'))\n                elif name == \'ResUNet\':\n                    roi, _, _ = extract_roi_from_binary_mask(with_images[index],\n                                    masks[index].astype(np.uint8), padding_fraction=.10)\n                    buffer = BytesIO()\n                    roi.save(buffer, format=\'JPEG\', quality=95)\n                    pixels = tf.io.decode_jpeg(buffer.getvalue(), channels=3)\n                    rois.append(tf.cast(tf.image.resize(pixels, (224, 224), method=\'bilinear\'),\n                                         tf.float32).numpy())\n                else:\n                    masked = seg_images[index] * masks[index][..., None].astype(np.float32)\n                    rois.append(tf.image.resize(masked, (224, 224)).numpy() * 255.0)\n            roi_batch = np.stack(rois).astype(np.float32)\n            if pipeline is not None:\n                scores = pipeline.classifier_probabilities(np.stack(full), roi_batch,\n                                                           batch_size=batch_size)\n            else:\n                scores = classifier.predict(roi_batch, batch_size=batch_size, verbose=0).ravel()\n            for index, row in enumerate(case_rows):\n                score = float(scores[index])\n                if not np.isfinite(score) or not 0 <= score <= 1:\n                    raise ValueError(f\'{name}: invalid model score on {row.image_id}\')\n                with Image.open(row.mask_path) as handle:\n                    truth = np.asarray(handle.convert(\'L\').resize((256, 256), Image.Resampling.NEAREST)) > 127\n                overlap = hard_overlap(truth, masks[index])\n                label = \'Melanoma\' if row.label == 1 else \'Benign\'\n                prediction = \'Melanoma\' if score >= threshold else \'Benign\'\n                rows.append({\'Pipeline\': name, \'Image ID\': row.image_id, \'True label\': label,\n                             \'Prediction\': prediction, \'Model score\': score,\n                             \'Decision threshold\': threshold, \'Correct\': prediction == label,\n                             \'Dice\': overlap[\'dice\'], \'IoU\': overlap[\'iou\']})\n            completed = min(start + batch_size, len(frame))\n            if start == 0 or completed % 40 < batch_size or completed == len(frame):\n                print(f\'{name}: {completed}/{len(frame)}\', flush=True)\n        table = pd.DataFrame(rows)\n        table.to_csv(csv_path, index=False)\n        return table\n    finally:\n        del pipeline, segmenter, classifier\n        tf.keras.backend.clear_session()\n        gc.collect()\n\n\ndef select_attention_advantages(all_results, min_dice=.90, min_lead=.005):\n    """A case must have correct Attention classification and a Dice lead over all peers."""\n    required = {\'Pipeline\', \'Image ID\', \'True label\', \'Prediction\', \'Dice\', \'IoU\'}\n    if not required.issubset(all_results.columns):\n        raise ValueError(\'Incomplete per-image comparison table.\')\n    if all_results.duplicated([\'Pipeline\', \'Image ID\']).any():\n        raise ValueError(\'Duplicate pipeline/image records.\')\n    rows = []\n    for image_id, group in all_results.groupby(\'Image ID\', sort=True):\n        if set(group[\'Pipeline\']) != set(ORDER) or len(group) != len(ORDER):\n            continue\n        if group[\'True label\'].nunique() != 1:\n            raise ValueError(\'Inconsistent reference diagnosis for \' + image_id)\n        attention = group[group[\'Pipeline\'] == \'Attention U-Net\'].iloc[0]\n        peers = group[group[\'Pipeline\'] != \'Attention U-Net\']\n        peer_best_dice = float(peers[\'Dice\'].max())\n        lead = float(attention[\'Dice\']) - peer_best_dice\n        correct = attention[\'Prediction\'] == attention[\'True label\']\n        peer_errors = int((peers[\'Prediction\'] != peers[\'True label\']).sum())\n        if (correct and float(attention[\'Dice\']) >= min_dice\n                and lead >= min_lead and float(attention[\'IoU\']) > float(peers[\'IoU\'].max())):\n            rows.append({\'Image ID\': image_id, \'Actual label\': attention[\'True label\'],\n                         \'Attention Dice\': float(attention[\'Dice\']),\n                         \'Best peer Dice\': peer_best_dice, \'Attention Dice lead\': lead,\n                         \'Peer classification errors\': peer_errors,\n                         \'Attention IoU\': float(attention[\'IoU\'])})\n    columns = [\'Image ID\', \'Actual label\', \'Attention Dice\', \'Best peer Dice\',\n               \'Attention Dice lead\', \'Peer classification errors\', \'Attention IoU\']\n    ranked = pd.DataFrame(rows, columns=columns)\n    if not ranked.empty:\n        # Prefer a classification advantage, then the largest segmentation lead.\n        ranked[\'classification_advantage\'] = ranked[\'Peer classification errors\'] > 0\n        ranked = ranked.sort_values([\'classification_advantage\', \'Attention Dice lead\',\n                                     \'Attention Dice\', \'Image ID\'],\n                                    ascending=[False, False, False, True])\n        ranked = ranked.drop(columns=\'classification_advantage\').reset_index(drop=True)\n    return ranked\n\n\ndef search_and_export(models, cases, output_dir, cache_dir, original_model_provenance,\n                      min_dice=.90, min_lead=.005, examples=3, batch_size=4):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    all_results = pd.concat([scan_pipeline(name, models[name], cases, cache_dir, batch_size)\n                             for name in ORDER], ignore_index=True)\n    all_results.to_csv(output_dir / \'all_models_all_searched_cases.csv\', index=False)\n    ranked = select_attention_advantages(all_results, min_dice, min_lead)\n    ranked.to_csv(output_dir / \'ranked_attention_advantages.csv\', index=False)\n    print(\'Cases meeting the declared rule:\', len(ranked), flush=True)\n    verified, rejected = [], []\n    for row in ranked.head(max(examples * 3, examples)).itertuples(index=False, name=None):\n        image_id = row[0]\n        source = cases[cases[\'image_id\'] == image_id].iloc[0]\n        true_label = \'Melanoma\' if int(source[\'label\']) == 1 else \'Benign\'\n        folder = output_dir / (\'case_\' + image_id)\n        reference = folder / \'scan_reference\'\n        reference.mkdir(parents=True, exist_ok=True)\n        comparison = all_results[all_results[\'Image ID\'] == image_id].copy()\n        comparison = comparison.rename(columns={\'Dice\': \'Hard Dice (this image)\',\n                                                \'IoU\': \'Hard IoU (this image)\'})\n        comparison.to_csv(reference / \'single_image_comparison.csv\', index=False)\n        print(\'Exporting and checking selected case:\', image_id, flush=True)\n        try:\n            _, fresh = export_all_stages(source[\'image_path\'], source[\'mask_path\'], models, folder,\n                previous_comparison=reference, source_manifest=original_model_provenance,\n                true_label=true_label,\n                selection_note=\'Selected Attention advantage: correct label and highest Dice\')\n            if select_attention_advantages(fresh, min_dice, min_lead).empty:\n                raise ValueError(\'Fresh single-image inference does not meet the selection rule.\')\n        except (ValueError, RuntimeError) as error:\n            rejected.append({\'Image ID\': image_id, \'Reason\': str(error)})\n            shutil.rmtree(folder)\n            continue\n        verified.append(image_id)\n        if len(verified) >= examples:\n            break\n    report = dict(selection_rule=dict(attention_correct=True, minimum_attention_dice=min_dice,\n                    minimum_dice_lead_over_every_peer=min_lead,\n                    preference=\'At least one peer classifier wrong, then largest Dice lead\'),\n                  number_of_searched_cases=len(cases), number_of_qualifying_cases=len(ranked),\n                  verified_exported_cases=verified, cases_rejected_after_verification=rejected,\n                  purpose=\'Selected illustrative examples; not an overall model ranking\',\n                  gradcam=\'Shown for the melanoma class. Not used as a selection metric.\',\n                  limitations=[\'Pipelines retain different training splits and ROI methods.\',\n                               \'The supplied U-Net++ notebook implements U-Net.\',\n                               \'The official test set has been examined during development.\'])\n    (output_dir / \'SELECTION_REPORT.json\').write_text(json.dumps(report, indent=2))\n    (output_dir / \'READ_ME.txt\').write_text(\n        \'These examples were deliberately selected to illustrate an Attention U-Net advantage.\\n\'\n        \'The selection rule and all searched per-image results are included.\\n\'\n        \'They do not establish overall superiority or clinical performance.\\n\'\n        \'The original weights, cutoffs and preprocessing were kept.\\n\'\n        \'A brighter or broader Grad-CAM is not evidence of a better prediction.\\n\'\n        \'Attention uses a padded RGB ROI plus the full image. U-Net masks the background \'\n        \'before cropping; the different classifier inputs are preserved.\\n\'\n        + (\'No case met the full rule; no winning picture was forced.\\n\' if not verified else\n           \'Open each case folder for the full nine-stage figure and two slide panels.\\n\'))\n    package = output_dir / \'Attention_UNet_Selected_Advantages.zip\'\n    with zipfile.ZipFile(package, \'w\', zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(output_dir.rglob(\'*\')):\n            if path.is_file() and path != package and path.suffix.lower() != \'.zip\':\n                archive.write(path, path.relative_to(output_dir))\n    return package, ranked, verified\n'}

for name, source in MODULES.items():
    (MODULE_DIR / name).write_text(source, encoding='utf-8')
sys.path.insert(0, str(MODULE_DIR))
importlib.invalidate_caches()
for name in ('attention_pipeline_core', 'bestcase_comparison_helpers',
             'full_stage_visuals', 'attention_case_search'):
    sys.modules.pop(name, None)
from full_stage_visuals import resolve_saved_models
from attention_case_search import prepare_cases, search_and_export

# Check the saved weights before starting any data download or model inference.
models, model_comparison, provenance = resolve_saved_models(
    DRIVE_ROOT, COMPARISON_ROOT, PREFERRED_COMPARISON)
labels = pd.DataFrame(LABELS)
cases = prepare_cases(labels, SEARCH_ROOT / 'test_cases', RUNTIME, scope=SEARCH_SCOPE)
print('Searching', len(cases), 'cases with saved weights and unchanged cutoffs.')
print('Attention must be correct, have Dice >=', MIN_ATTENTION_DICE,
      'and exceed every peer Dice by at least', MIN_DICE_LEAD)
print('Completed model searches are cached for reruns. No training occurs.')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_DIR = SEARCH_ROOT / 'results' / RUN_ID
package, ranked, verified = search_and_export(
    models, cases, OUTPUT_DIR, SEARCH_ROOT / 'scan_cache', provenance,
    min_dice=MIN_ATTENTION_DICE, min_lead=MIN_DICE_LEAD,
    examples=NUMBER_OF_EXAMPLES, batch_size=4)
display(ranked.head(10).round(4))
if verified:
    print('Verified examples:', ', '.join(verified))
    first = OUTPUT_DIR / ('case_' + verified[0])
    for name in ('FULL_COMPARISON_ALL_STAGES.png', 'SLIDE_A_SEGMENTATION_COMPARISON.png',
                 'SLIDE_B_ROI_AND_GRADCAM_COMPARISON.png'):
        print(verified[0], name)
        display(DisplayImage(filename=str(first / name), width=1600))
    print('Other examples and individual stage PNGs are in the downloaded ZIP.')
else:
    print('No case met the full rule. The ZIP contains the search tables and report, not a forced winner.')
print('Saved outputs:', OUTPUT_DIR)
files.download(str(package))
